In [1]:
import os
import pandas as pd
import numpy as np

print(os.listdir("data"))

['Sample - Superstore.csv']


In [2]:
df = pd.read_csv("data/Sample - Superstore.csv", encoding="latin-1")
print(df.shape)
print(df.dtypes)
df.head(3)

(9994, 21)
Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code        int64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


In [3]:
print("missing cells:", int(df.isnull().sum().sum()))
print("full duplicate rows:", df.duplicated().sum())
print("Row ID unique:", df["Row ID"].is_unique)
print("duplicate Order ID + Product ID:", df.duplicated(["Order ID", "Product ID"]).sum())
print("Product IDs with more than one name:",
      int((df.groupby("Product ID")["Product Name"].nunique() > 1).sum()))
print("Customer IDs with more than one name or segment:",
      int((df.groupby("Customer ID")[["Customer Name", "Segment"]].nunique() > 1).any(axis=1).sum()))

print("\nPostal Code length (4 = a leading zero was lost):")
print(df["Postal Code"].astype(str).str.len().value_counts())

print("\n", df[["Sales", "Quantity", "Discount", "Profit"]].describe().round(2))
print("\nDiscount levels:\n", df["Discount"].value_counts().sort_index())
print("loss-making rows:", int((df["Profit"] < 0).sum()))

missing cells: 0
full duplicate rows: 0
Row ID unique: True
duplicate Order ID + Product ID: 8
Product IDs with more than one name: 32
Customer IDs with more than one name or segment: 0

Postal Code length (4 = a leading zero was lost):
Postal Code
5    9545
4     449
Name: count, dtype: int64

           Sales  Quantity  Discount   Profit
count   9994.00   9994.00   9994.00  9994.00
mean     229.86      3.79      0.16    28.66
std      623.25      2.23      0.21   234.26
min        0.44      1.00      0.00 -6599.98
25%       17.28      2.00      0.00     1.73
50%       54.49      3.00      0.20     8.67
75%      209.94      5.00      0.20    29.36
max    22638.48     14.00      0.80  8399.98

Discount levels:
 Discount
0.00    4798
0.10      94
0.15      52
0.20    3657
0.30     227
0.32      27
0.40     206
0.45      11
0.50      66
0.60     138
0.70     418
0.80     300
Name: count, dtype: int64
loss-making rows: 1871


In [4]:
clean = df.copy()

for col in clean.columns:
    if pd.api.types.is_string_dtype(clean[col]):
        clean[col] = clean[col].str.strip()

clean["Order Date"] = pd.to_datetime(clean["Order Date"], format="%m/%d/%Y")
clean["Ship Date"] = pd.to_datetime(clean["Ship Date"], format="%m/%d/%Y")
clean["Postal Code"] = clean["Postal Code"].astype(str).str.zfill(5)

clean.columns = [c.lower().replace(" ", "_").replace("-", "_") for c in clean.columns]

ship_days = (clean["ship_date"] - clean["order_date"]).dt.days
print("order dates:", clean["order_date"].min().date(), "to", clean["order_date"].max().date())
print("shipping days min/max:", ship_days.min(), ship_days.max())
print("postal code lengths:", clean["postal_code"].str.len().unique())
print(list(clean.columns))

order dates: 2014-01-03 to 2017-12-30
shipping days min/max: 0 7
postal code lengths: [5]
['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


In [5]:
dim_product = (clean[["product_id", "product_name", "category", "sub_category"]]
               .drop_duplicates().reset_index(drop=True))
dim_product.insert(0, "product_key", dim_product.index + 1)

dim_customer = (clean[["customer_id", "customer_name", "segment"]]
                .drop_duplicates().reset_index(drop=True))
dim_customer.insert(0, "customer_key", dim_customer.index + 1)

dim_location = (clean[["country", "region", "state", "city", "postal_code"]]
                .drop_duplicates().reset_index(drop=True))
dim_location.insert(0, "location_key", dim_location.index + 1)

fact = (clean
        .merge(dim_product, on=["product_id", "product_name", "category", "sub_category"], how="left")
        .merge(dim_customer, on=["customer_id", "customer_name", "segment"], how="left")
        .merge(dim_location, on=["country", "region", "state", "city", "postal_code"], how="left"))

fact = fact[["row_id", "order_id", "order_date", "ship_date", "ship_mode",
             "customer_key", "product_key", "location_key",
             "sales", "quantity", "discount", "profit"]]

years = clean["order_date"].dt.year
dates = pd.date_range(f"{years.min()}-01-01", f"{years.max()}-12-31", freq="D")
dim_date = pd.DataFrame({"date": dates})
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = "Q" + dim_date["date"].dt.quarter.astype(str)
dim_date["month_num"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name().str[:3]
dim_date["year_month"] = dim_date["date"].dt.strftime("%Y-%m")
dim_date["weekday"] = dim_date["date"].dt.day_name()

print({"fact": len(fact), "products": len(dim_product), "customers": len(dim_customer),
       "locations": len(dim_location), "dates": len(dim_date)})

{'fact': 9994, 'products': 1894, 'customers': 793, 'locations': 632, 'dates': 1461}


In [6]:
print("fact rows == raw rows:", len(fact) == len(df))
print("missing keys in fact:", int(fact[["customer_key", "product_key", "location_key"]].isnull().sum().sum()))
print("keys unique:", dim_product["product_key"].is_unique,
      dim_customer["customer_key"].is_unique, dim_location["location_key"].is_unique)

print("sales  raw vs fact:", round(df["Sales"].sum(), 2), round(fact["sales"].sum(), 2))
print("profit raw vs fact:", round(df["Profit"].sum(), 2), round(fact["profit"].sum(), 2))
print("units  raw vs fact:", int(df["Quantity"].sum()), int(fact["quantity"].sum()))
print("fact order dates outside the date table:",
      int((~fact["order_date"].isin(dim_date["date"])).sum()))

fact rows == raw rows: True
missing keys in fact: 0
keys unique: True True True
sales  raw vs fact: 2297200.86 2297200.86
profit raw vs fact: 286397.02 286397.02
units  raw vs fact: 37873 37873
fact order dates outside the date table: 0


In [7]:
os.makedirs("data/model", exist_ok=True)
for name, table in {"fact_sales": fact, "dim_product": dim_product, "dim_customer": dim_customer,
                    "dim_location": dim_location, "dim_date": dim_date}.items():
    table.to_csv(f"data/model/{name}.csv", index=False, encoding="utf-8-sig")
print(os.listdir("data/model"))

['dim_customer.csv', 'dim_date.csv', 'dim_location.csv', 'dim_product.csv', 'fact_sales.csv']


In [8]:
print("unique product IDs:", df["Product ID"].nunique(), "| unique names:", df["Product Name"].nunique(),
      "| product rows in dimension:", len(dim_product))

bands = pd.cut(df["Discount"], [-0.01, 0, 0.2, 0.5, 1.0],
               labels=["none", "up to 20%", "21 to 50%", "over 50%"])
t = df.groupby(bands, observed=True).agg(rows=("Profit", "size"), sales=("Sales", "sum"), profit=("Profit", "sum"))
t["margin_pct"] = (t["profit"] / t["sales"] * 100).round(1)
print(t.round(0))

unique product IDs: 1862 | unique names: 1850 | product rows in dimension: 1894
           rows      sales    profit  margin_pct
Discount                                        
none       4798  1087908.0  320988.0        30.0
up to 20%  3803   846522.0  100785.0        12.0
21 to 50%   537   298541.0  -58817.0       -20.0
over 50%    856    64229.0  -76559.0      -119.0


In [9]:
print(os.listdir("data/model"))

['dim_customer.csv', 'dim_date.csv', 'dim_location.csv', 'dim_product.csv', 'fact_sales.csv']
